# Forecast baseline y punto de reorden

Esta notebook genera la tabla de decisión por SKU:

1. Demanda diaria higienizada (`prepare_daily_demand`)
2. Clasificación ABC del último trimestre
3. Features rolling y backtest de la media móvil 30 días
4. Punto de reorden, stock de seguridad y pedido sugerido

El detalle de por qué se rellenan ceros y se recortan picos está en la notebook 03.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "inventario_ecommerce").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from inventario_ecommerce import config
from inventario_ecommerce.dataset import load_transactions, save_processed
from inventario_ecommerce.features import (
    clean_transactions,
    compute_abc_classification,
    prepare_daily_demand,
    build_rolling_features,
    sales_by_product_last_quarter,
)
from inventario_ecommerce.modeling.predict import build_reorder_policy, forecast_30d_baseline
from inventario_ecommerce.modeling.train import temporal_backtest_baseline

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)


## 1. Datos y demanda diaria

La serie de cada SKU incluye los días sin venta (cantidad = 0) desde su primera transacción. Sin eso, la media solo vería días con ticket y sesgaría la demanda al alza.


In [ ]:
raw = load_transactions()
clean = clean_transactions(raw)
daily = prepare_daily_demand(clean)
rolling = build_rolling_features(daily)

latest_features = (
    rolling.sort_values("Date")
    .groupby([config.COL_STOCK_CODE, config.COL_DESCRIPTION], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

print(f"Filas raw:     {len(raw):,}")
print(f"Filas limpias: {len(clean):,}")
print(f"Días-SKU:      {len(daily):,}")
print(f"SKUs:          {latest_features[config.COL_STOCK_CODE].nunique():,}")


## 2. Clasificación ABC

- **A**: hasta el 80% de las ventas acumuladas
- **B**: del 80% al 95%
- **C**: el resto


In [ ]:
last_q = sales_by_product_last_quarter(clean)
abc = compute_abc_classification(last_q)
abc["ABCClass"].value_counts()


## 3. Backtest temporal

Holdout: últimos 30 días. Predicción: media diaria de los 30 días anteriores al corte. MAE y MAPE se calculan sobre el calendario completo (incluye ceros).


In [ ]:
sku_metrics, global_metrics = temporal_backtest_baseline(
    daily, horizon_days=30, lookback_days=30
)
global_metrics


## 4. Política de reorden

Supuestos del baseline (el dataset no trae stock ni lead time real):

| Parámetro | Valor |
|-----------|------:|
| Lead time | 14 días |
| Ciclo de revisión | 7 días |
| z clase A / B / C | 1.88 / 1.65 / 1.28 |

`ROP = forecast_daily × LT + z × σ_30d × √LT`

`recommended_order_qty` es la demanda del ciclo de revisión (`forecast_daily × 7`).


In [ ]:
forecast = forecast_30d_baseline(daily, lookback_days=30, horizon_days=30)
policy = build_reorder_policy(latest_features, forecast, abc)
policy.head(20)


## 5. Artefactos


In [ ]:
save_processed(abc, "abc_last_quarter.csv")
save_processed(latest_features, "sku_rolling_features_latest.csv")
save_processed(sku_metrics, "forecast_backtest_by_sku.csv")
save_processed(global_metrics, "forecast_backtest_global.csv")
save_processed(policy, "inventory_reorder_recommendations.csv")

print("SKUs en la tabla de reorden:", f"{len(policy):,}")
print("Carpeta:", config.PROCESSED_DATA_DIR)


Salida principal: `data/processed/inventory_reorder_recommendations.csv`.

Campos clave: `ABCClass`, `forecast_30d`, `safety_stock`, `reorder_point`, `target_stock`, `recommended_order_qty`.
